In [11]:
import json

In [2]:
pi      = 3.14159265359 #: Traditionally only used 12 proint precision for this script

In [5]:
def linear_interpolate(x1, y1, x2, y2, xin):
    """return y coordinate corresponding to xin along the line define by (x1, y1)/(x2, y2)

    :param x1: x-coord of point 1
    :type x1: float
    :param y1: y-coord of point 1
    :type y1: float
    :param x2: x-coord of point 2
    :type x2: float
    :param y2: y-coord of point 2
    :type y2: float
    :param xin: x value to be estimated
    :type xin: float
    :return: y coordinate corresponding to xin
    :rtype: float
    """
    y = y1 - ((y1 - y2) * (x1 - xin)) / (x1 - x2)
    return y

In [4]:
def solwind(xkp):
    """
    get the solar wind parameters used as inputs for the bow shock
    and magnetopause boundary models.
    
    input:  xkp     --- kp index (real value between 0 & 9).
    output: bx      --- the imf b_x [nt]
            by      --- the imf b_y [nt]
            bz      --- the imf b_z [nt]
            vx      --- x component of solar wind bulk flow velocity (km/s).
            vy      --- y component of solar wind bulk flow velocity (km/s).
            vz      --- z component of solar wind bulk flow velocity (km/s).
            dennum  --- the solar wind proton number density [#/cm^3]
            swetemp --- the solar wind electron temperature [k]
            swptemp --- the solar wind proton temperature [k]
            hefrac  --- fraction of solar wind ions which are helium ions
            swhtemp --- the temperature of the helium [k]
            bowang  --- angle bow shock radius calculated (rad).
            dypres  --- solar wind dynamic pressure (np).
            abang   --- aberration angle of magnetotail (deg).
            xhinge  --- hinge point of magnetotail (re).

    from parm file      pi
    """

    bx      =   -5.0
    by      =    6.0
    bz      =    6.0
    vx      = -500.0
    vy      =    0.0
    vz      =    0.0

    dennum  = 8.0
    swetemp = 1.4e+5
    swptemp = 1.2e+5
    hefrac  = 0.047
    swhtemp = 5.8e+5
    bowang  = pi

    if xkp <= 4.0:
        dypres  =    1.0
        abang   =    4.0
        xhinge  =   14.0
        vx      = -400.0

    elif (xkp > 4.0) and (xkp <= 6.0): 
        dypres1 =    1.0
        abang1  =    4.0
        vx1     = -400.0
        dypres2 =    4.0
        abang2  =    0.0
        vx2     = -500.0

        dypres  = linear_interpolate(4.0, dypres1, 6.0, dypres2, xkp)
        abang   = linear_interpolate(4.0, abang1,  6.0, abang2, xkp)
        xhinge  = 14.0
        vx      = linear_interpolate(4.0, vx1, 6.0, vx2, xkp)

    else:
        dypres1 =    4.0
        dypres2 =   10.0
        dypres  = linear_interpolate(6.0, dypres1, 9.0, dypres2, xkp)
        abang   =    0.0
        xhinge  =   14.0
        vx      = -500.0

    return [bx, by, bz, vx, vy, vz, dennum, swetemp, swptemp, hefrac,\
            swhtemp, bowang, dypres, abang, xhinge]

In [9]:
solar_wind_parameters = {}
for i in range(0, 28):
    xkp   = i / 3.0
    key_xkp = f"{xkp:.2f}"
    [bx, by, bz, vx, vy, vz, dennum, swetemp, swptemp, hefrac, swhtemp, bowang, dypres, abang, xhinge] = solwind(xkp)
    solar_wind_parameters[key_xkp] = {
        'bx':bx,
        'by':by,
        'bz':bz,
        'vx':vx,
        'vy':vy,
        'vz':vz,
        'dennum':dennum,
        'swetemp':swetemp,
        'swptemp':swptemp,
        'hefrac':hefrac,
        'swhtemp':swhtemp,
        'bowang':bowang,
        'dypres':dypres,
        'abang':abang,
        'xhinge':xhinge
    }

In [12]:
with open('solar_wind_parameters.json','w') as f:
    json.dump(solar_wind_parameters,f)